# WTI Review Runner

Run the company review sequence for both daily and weekly WTI experiments.

Default batch order:

1. Daily report setting requested for the company update
2. Daily scaled + regularized improvement run
3. Weekly report setting requested for the company update
4. Weekly scaled + regularized improvement run

If needed, you can add the raw reference or validation-size sweep by uncommenting them in `BATCH_CONFIG_RELATIVE_PATHS`.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

RUNTIME_SENTINEL = Path("/content/newoil_env_ready_v2")

REQUIRED_PACKAGES = [
    "numpy==2.1.3",
    "pandas==2.2.3",
    "matplotlib==3.10.1",
    "openpyxl==3.1.5",
    "pyyaml==6.0.2",
    "neuralforecast==3.1.7",
]
if not RUNTIME_SENTINEL.exists():
    subprocess.run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        *REQUIRED_PACKAGES,
    ], check=True)
    RUNTIME_SENTINEL.write_text("ready\n", encoding="utf-8")
    print("Dependencies installed. Restarting the Colab runtime once to load clean binary wheels.")
    os.kill(os.getpid(), 9)

import pandas as pd

REPO_URL = "https://github.com/Jaeho777/newoil.git"
WORKDIR = Path("/content/newoil")

# If you received an updated weekly CSV, put it in Google Drive and set the full path below.
# Example: Path("/content/drive/MyDrive/newoil_inputs/0428DB_weekly.csv")
UPDATED_WEEKLY_DATA_SOURCE_PATH = None

BATCH_CONFIG_RELATIVE_PATHS = [
    "configs/batches/daily_wti_h12_mse_report.yaml",
    "configs/batches/daily_wti_h12_mse_scaled_regularized.yaml",
    "configs/batches/weekly_wti_h2_mse_report.yaml",
    "configs/batches/weekly_wti_h2_mse_scaled_regularized.yaml",
    # "configs/batches/weekly_wti_h2_mse_raw_report.yaml",
    # "configs/batches/weekly_wti_h2_mse_scaled_val_sweep.yaml",
]

SAVE_TO_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/newoil_outputs")
LOCAL_OUTPUT_ROOT = Path("/content/newoil_outputs")

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)
sys.path.insert(0, str(WORKDIR / "src"))

for module_name in list(sys.modules):
    if module_name == "newoil" or module_name.startswith("newoil."):
        del sys.modules[module_name]

from newoil import build_company_master_report, run_batch_from_config

repo_root = WORKDIR
output_root = DRIVE_OUTPUT_ROOT if SAVE_TO_GOOGLE_DRIVE else LOCAL_OUTPUT_ROOT
output_root.mkdir(parents=True, exist_ok=True)

if UPDATED_WEEKLY_DATA_SOURCE_PATH:
    source_path = Path(UPDATED_WEEKLY_DATA_SOURCE_PATH)
    target_path = repo_root / "data" / "0428DB_weekly.csv"
    print(f"Replacing weekly data: {source_path} -> {target_path}")
    shutil.copy2(source_path, target_path)
else:
    print("Using weekly data committed in the repository.")

all_summaries = []
batch_dirs = []
batch_results = []
for relative_path in BATCH_CONFIG_RELATIVE_PATHS:
    batch_config_path = repo_root / relative_path
    print(f"\n[RUN BATCH] {batch_config_path}")
    result = run_batch_from_config(
        batch_config_path=batch_config_path,
        repo_root=repo_root,
        output_root=output_root,
    )
    batch_results.append(result)
    summary_df = result.summary_df.copy()
    summary_df["batch_name"] = Path(relative_path).name
    all_summaries.append(summary_df)
    batch_dirs.append(str(result.batch_dir))

combined_summary_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()
combined_summary_path = output_root / "wti_review_combined_summary.csv"
combined_summary_df.to_csv(combined_summary_path, index=False)
master_report_path = build_company_master_report(batch_results, output_root)

print("\nBatch directories:")
for batch_dir in batch_dirs:
    print(batch_dir)

print(f"\nCombined summary: {combined_summary_path}")
print(f"Master report: {master_report_path}")
combined_summary_df
